### Setup

In [1]:
%load_ext autoreload
%autoreload 2

import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

from app.config import get_settings

settings = get_settings()
os.environ["HF_TOKEN"] = settings.hf_token

print({k: ("set" if v else "MISSING") for k, v in settings.model_dump().items()})

{'app_env': 'set', 'database_url': 'set', 'groq_api_key': 'set', 'groq_model': 'set', 'hf_token': 'set', 'embedding_model': 'set', 'fernet_key': 'set', 'llm_cache_dir': 'set', 'mem0_api_key': 'set'}


In [2]:
from sqlalchemy import create_engine, text

engine = create_engine(settings.database_url, pool_pre_ping=True)


def sql(query: str, **params) -> pd.DataFrame:
    """Run a query on the app database and return a DataFrame."""
    with engine.connect() as conn:
        res = conn.execute(text(query), params)
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


display(sql("select version()"))
display(sql("select extname, extversion from pg_extension where extname = 'vector'"))

,version
0,"PostgreSQL 17.11 on aarch64-unknown-linux-gnu,..."


,extname,extversion
0,vector,0.8.2


In [4]:
from groq import APIConnectionError, Groq, RateLimitError

client = Groq(api_key=settings.groq_api_key)

In [5]:
CACHE_DIR = Path(settings.llm_cache_dir)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def llm(messages, *, model=None, temperature=0.0, json_mode=False, use_cache=True, **kwargs) -> str:
    """Single entry point for all LLM calls. Cached on disk so reruns are free and reproducible."""
    model = model or settings.groq_model
    payload = {"model": model, "messages": messages, "temperature": temperature,
               "json_mode": json_mode, **kwargs}
    key = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    path = CACHE_DIR / f"{key}.json"
    if use_cache and path.exists():
        return json.loads(path.read_text(encoding="utf-8"))["text"]

    params = dict(model=model, messages=messages, temperature=temperature, **kwargs)
    if json_mode:
        params["response_format"] = {"type": "json_object"}

    for attempt in range(4):
        try:
            resp = client.chat.completions.create(**params)
            break
        except (RateLimitError, APIConnectionError):
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)

    text_out = resp.choices[0].message.content
    path.write_text(json.dumps({"text": text_out}), encoding="utf-8")
    return text_out


msgs = [{"role": "user", "content": "Reply with exactly one word: ready"}]
for attempt in (1, 2):
    t0 = time.perf_counter()
    print(attempt, llm(msgs), f"{time.perf_counter() - t0:.2f}s")

1 ready 0.52s
2 ready 0.00s


In [6]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings

_embedder = HuggingFaceEndpointEmbeddings(model=settings.embedding_model)

EMB_DIR = Path(settings.llm_cache_dir).parent / "emb"
EMB_DIR.mkdir(parents=True, exist_ok=True)


def _emb_path(text: str) -> Path:
    key = hashlib.sha256(f"{settings.embedding_model}\n{text}".encode()).hexdigest()
    return EMB_DIR / f"{key}.json"


def embed(texts: list[str], *, use_cache: bool = True, batch_size: int = 32) -> list[list[float]]:
    """Embed texts, caching each one on disk so reruns are free and reproducible."""
    out = [None] * len(texts)
    missing = []
    for i, t in enumerate(texts):
        p = _emb_path(t)
        if use_cache and p.exists():
            out[i] = json.loads(p.read_text(encoding="utf-8"))
        else:
            missing.append(i)
    for s in range(0, len(missing), batch_size):
        idx = missing[s:s + batch_size]
        vecs = _embedder.embed_documents([texts[i] for i in idx])
        for i, v in zip(idx, vecs):
            out[i] = v
            _emb_path(texts[i]).write_text(json.dumps(v), encoding="utf-8")
    return out


def embed_query(text: str) -> list[float]:
    return embed([text])[0]


def cos(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


vecs = embed(["customer orders", "product price", "customer purchases"])
print("vectors:", len(vecs), "| dimension:", len(vecs[0]))
print("orders vs purchases:", round(cos(vecs[0], vecs[2]), 3))
print("orders vs price:    ", round(cos(vecs[0], vecs[1]), 3))

ConnectError: [WinError 10054] An existing connection was forcibly closed by the remote host